[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h03_longitude.ipynb)

# Find your longitude with a clock

For centuries a ship could measure how far north it was, but not how far east. The answer was a clock. The Earth
turns 360 degrees in 24 hours, 15 degrees every hour, so the gap between your noon and London's noon is your
longitude. Here you can find your own longitude from a clock, see how fast a clock's small daily error turns into
miles at sea, and watch the Moon creep across the stars like the hand of a clock.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

DEG_PER_HOUR = 360 / 24                     # 15 degrees an hour
MIN_PER_DEG = 24 * 60 / 360                 # 4 minutes a degree
KM_PER_DEG = 2 * math.pi * 6378.137 / 360   # one degree of longitude at the equator, about 111 km

print("Ready. Run the next cell.")

## 1 · Your noon, London's clock

You carry a clock set to London time. Far out at sea, the Sun reaches its highest point: that's noon where you are.
Set the London clock's reading at that moment. Every hour behind London is 15 degrees west; every hour ahead is 15
degrees east.

In [ ]:
def longitude(hour=15, minute=0):
    gap = (hour * 60 + minute) - 12 * 60           # minutes the London clock is past noon
    deg = gap / MIN_PER_DEG
    side = "west" if deg > 0 else "east" if deg < 0 else ""
    print(f"London clock at your noon: {hour:02d}:{minute:02d}")
    print(f"you are {abs(gap) // 60} h {abs(gap) % 60} min {'behind' if gap > 0 else 'ahead of'} London")
    print(f"longitude: {abs(deg):g}° {side}".rstrip())
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)     # seen from above the pole, west is clockwise
    ax.set_yticks([]); ax.set_xticks([])
    th = [math.radians(k) for k in range(361)]
    ax.plot(th, [1] * 361, color=DIM)
    ax.plot([0, 0], [0, 1], color=CHALK, lw=2); ax.text(0, 1.12, "London", ha="center", color=CHALK)
    ax.plot([math.radians(deg)] * 2, [0, 1], color=CORAL, lw=3); ax.text(math.radians(deg), 1.12, "you", ha="center", color=CORAL)
    ax.set_title("looking down on the North Pole: the meridians", pad=24); plt.show()

interact(longitude, hour=IntSlider(value=15, min=0, max=23, description="hour"),
         minute=IntSlider(value=0, min=0, max=59, description="minute"));

## 2 · A clock that drifts

Every 4 minutes of error costs a whole degree, about 111 km at the equator. Set how many seconds your clock gains or
loses each day, and how long the voyage is. The prize of 1714 asked for half a degree: 2 minutes of time.

In [ ]:
def drift(seconds_a_day=3.0, days=42):
    off_s = seconds_a_day * days
    deg = off_s / 60 / MIN_PER_DEG
    print(f"after {days} days: {off_s:.0f} s off  →  {deg:.2f}° of longitude  →  about {deg * KM_PER_DEG:.0f} km at the equator")
    print("inside the prize's half a degree" if deg <= 0.5 else "outside the prize's half a degree")
    d = list(range(days + 1))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(d, [seconds_a_day * k for k in d], color=TEAL, lw=2.5, label="your clock's error")
    ax.axhline(120, color=CORAL, lw=2, label="2 minutes = half a degree")
    ax.set_xlabel("days at sea"); ax.set_ylabel("seconds off"); ax.legend(); plt.show()

interact(drift, seconds_a_day=FloatSlider(value=3.0, min=0.0, max=15.0, step=0.1, description="s a day"),
         days=IntSlider(value=42, min=1, max=120, description="days"));

## 3 · Harrison's watch at Jamaica

In 1761–62, Harrison's watch H4 sailed 81 days to Jamaica and arrived 5 seconds slow. How far off does that put a
ship? One nautical mile is one arcminute along a great circle, and at a latitude φ a degree of longitude shrinks by
cos φ. Jamaica is at about 18° north.

In [ ]:
def h4(seconds_off=5, latitude=18):
    arcmin = seconds_off / (MIN_PER_DEG * 60) * 60
    nm = arcmin * math.cos(math.radians(latitude))
    print(f"{seconds_off} s of time = {arcmin:.2f} arcminutes of longitude")
    print(f"at {latitude}° north that is about {nm:.2f} nautical miles ({nm * 1.852:.1f} km)")
    print(f"the prize allowed 120 s: this is {seconds_off / 120:.1%} of it")

interact(h4, seconds_off=IntSlider(value=5, min=0, max=240, description="seconds off"),
         latitude=IntSlider(value=18, min=0, max=80, description="latitude"));

## 4 · The Moon as a clock hand

The astronomers' idea: the Moon moves across the stars, once round in about 27.3 days. Drag the hours and watch it
slide past a star. It moves about its own width every hour.

In [ ]:
MOON_DEG_H = 360 / (27.321661 * 24)
MOON_WIDTH = math.degrees(2 * math.atan(1737.4 / 384_400))

def moon(hours=0.0):
    x = -2 + MOON_DEG_H * hours
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.scatter([0.6, 2.2, -1.1, 3.1], [0.3, -0.4, -0.2, 0.5], marker="*", s=[220, 90, 70, 120], color=CHALK)
    ax.add_patch(plt.Circle((x, 0), MOON_WIDTH / 2, color=LILAC))
    ax.annotate("", xy=(0.6, 0.3), xytext=(x, 0), arrowprops={"arrowstyle": "-", "color": CORAL, "lw": 2})
    ax.set_xlim(-3, 4); ax.set_ylim(-1, 1); ax.set_aspect("equal"); ax.set_xlabel("degrees across the sky")
    ax.set_title(f"after {hours:.1f} h the Moon has moved {MOON_DEG_H * hours:.2f}°  (it is {MOON_WIDTH:.2f}° wide)")
    plt.show()
    print(f"angle from the Moon's centre to the bright star: {math.hypot(0.6 - x, 0.3):.2f}°")

interact(moon, hours=FloatSlider(value=0.0, min=0.0, max=8.0, step=0.25, description="hours"));

## 5 · Clocks in the sky

GPS satellites carry atomic clocks. Your phone turns each signal's delay into a distance, at the speed of light. A
clock error of a billionth of a second (a nanosecond) is about 30 cm.

In [ ]:
C = 299_792_458                              # metres a second

def gps(nanoseconds=1):
    print(f"{nanoseconds} ns of clock error  →  {C * nanoseconds * 1e-9:.2f} m of distance")

interact(gps, nanoseconds=IntSlider(value=1, min=1, max=1000, description="ns off"));

## 6 · The challenge from the video

You're at sea, and the Sun has just reached its highest point. Your clock, still set to London time, says twenty past
four in the afternoon. How far west of London are you? Think first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("London clock at your noon: 16:20. How far west?   (press the button when you're ready)"); return
    gap = 4 * 60 + 20
    print(f"4 h 20 min behind London = {gap} minutes")
    print(f"{gap} ÷ 4 minutes a degree = {gap / MIN_PER_DEG:g}° west")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Royal Museums Greenwich: the Longitude Act; Newton has his say. Wikipedia: John Harrison; Longitude Act; Scilly naval
disaster of 1707; Lunar distance (navigation); Nevil Maskelyne; Larcum Kendall. MacTutor: Tobias Mayer. R. Dunn and
R. Higgitt, *Finding Longitude* (2014). All retrieved 2026-10-28.